# XOR Gate Model

In [9]:
import numpy as np
from qutip import basis, tensor, sigmaz, expect, qeye
from qutip.qip.operations import rx, cnot
from scipy.optimize import minimize
from circuit_maker import create_xor_circuit_text

### XOR Training Set

In [10]:
training_data = {
    (0, 0): 0,
    (0, 1): 1,
    (1, 0): 1,
    (1, 1): 0
}

### Defining P1 Projector

In [11]:
P1 = basis(2, 1) * basis(2, 1).dag()
Proj_q1 = tensor(qeye(2), P1)

<h3>Quantum Circuit</h3>
<img src="xor_symbolic_circuit.png" alt="XOR Quantum Circuit" width="300"/>

In [12]:
def circuit_expectation(params, bits):
    x1, x2 = bits
    state = tensor(basis(2, x1), basis(2, x2))
    
    # Ansatz layers
    U1 = tensor(rx(params[0]), rx(params[1]))
    U2 = cnot(2, 0, 1)
    U3 = tensor(rx(params[2]), rx(params[3]))
    U = U3 * U2 * U1
    
    final = U * state
    
    return expect(Proj_q1, final).real

### Loss Function

In [13]:
def loss(params):
    errs = []
    for bits, target in training_data.items():
        e = circuit_expectation(params, bits)
        errs.append((e - target)**2)
    return np.mean(errs)


### Optimizating Parameters

In [14]:
# Optimize
init = np.random.uniform(0, 2*np.pi, 4)
result = minimize(loss, init, bounds=[[0, 2*np.pi], [0, 2*np.pi], [0, 2*np.pi], [0, 2*np.pi]])
opt_params = result.x

qc = create_xor_circuit_text(np.round(opt_params, 3))

# Report
print("Learned params:", np.round(opt_params, 3))

print(f"\nQuantum Circuit:")
print(qc)

Learned params: [3.171 5.405 0.622 4.02 ]

 Quantum Circuit:
     ┌───────────┐     ┌───────────┐            
q_0: ┤ Rx(3.171) ├──■──┤ Rx(0.622) ├────────────
     ├───────────┤┌─┴─┐└┬──────────┤┌────┐┌────┐
q_1: ┤ Rx(5.405) ├┤ X ├─┤ Rx(4.02) ├┤ P1 ├┤ P1 ├
     └───────────┘└───┘ └──────────┘└────┘└────┘


### Result

In [19]:
for bits, target in training_data.items():
    e = circuit_expectation(opt_params, bits)
    pred = int(round(e))
    print(f"{bits} → P(1)={e:.3f}, pred={pred}, target={target}")

(0, 0) → P(1)=0.000, pred=0, target=0
(0, 1) → P(1)=1.000, pred=1, target=1
(1, 0) → P(1)=1.000, pred=1, target=1
(1, 1) → P(1)=0.000, pred=0, target=0


### Optimizated Operator

In [16]:
U1 = tensor(rx(opt_params[0]), rx(opt_params[1]))
U2 = cnot(2, 0, 1)
U3 = tensor(rx(opt_params[2]), rx(opt_params[3]))
U = U3 * U2 * U1

U

Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=Dense, isherm=False
Qobj data =
[[-2.18066401e-06-3.05798283e-01j -4.77727622e-05-1.39586509e-02j
  -4.48382962e-03-1.48721822e-04j  9.51983423e-01+7.00477858e-07j]
 [-4.77727622e-05-1.39586509e-02j -2.18066401e-06-3.05798283e-01j
   9.51983423e-01+7.00477858e-07j -4.48382962e-03-1.48721822e-04j]
 [ 9.51983423e-01+7.00477858e-07j -4.48382962e-03-1.48721822e-04j
  -4.77727622e-05-1.39586509e-02j -2.18066401e-06-3.05798283e-01j]
 [-4.48382962e-03-1.48721822e-04j  9.51983423e-01+7.00477858e-07j
  -2.18066401e-06-3.05798283e-01j -4.77727622e-05-1.39586509e-02j]]